In [2]:
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

from scipy.special import logit, expit
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score
import pandas as pd


import random
import yaml
from pathlib import Path
import time
import datetime
from string import Template
import os
import json
#from ..dataset.abstract_dataset import DeepfakeAbstractBaseDataset

from datasets.abstract_dataset import DeepfakeAbstractBaseDataset
from detectors import DETECTOR


#from metrics.utils import get_test_metrics

In [3]:
def load_config(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)
    
def init_seed(config):
    if config['manualSeed'] is None:
        config['manualSeed'] = random.randint(1, 10000)
    random.seed(config['manualSeed'])
    torch.manual_seed(config['manualSeed'])
    if config['cuda']:
        torch.cuda.manual_seed_all(config['manualSeed'])

In [4]:
# Adapted from test.py

def prepare_testing_data(config):
    def get_test_data_loader(config, test_name):
        # update the config dictionary with the specific testing dataset
        config = config.copy()  # create a copy of config to avoid altering the original one
        config['test_dataset'] = test_name  # specify the current test dataset
        test_set = DeepfakeAbstractBaseDataset(
                config=config,
                mode='test', 
            )
        
        test_data_loader = \
            torch.utils.data.DataLoader(
                dataset=test_set, 
                batch_size=config['test_batchSize'],
                shuffle=False, 
                num_workers=int(config['workers']),
                collate_fn=test_set.collate_fn,
                drop_last=False
            )
        return test_data_loader

    test_data_loaders = {}
    for one_test_name in config['test_dataset']:
        test_data_loaders[one_test_name] = get_test_data_loader(config, one_test_name)
    return test_data_loaders

In [5]:
@torch.no_grad()
def inference(model, data_dict):
    predictions = model(data_dict, inference=True)
    return predictions

def test_one_dataset(model, data_loader, device):
    prediction_lists = []
    logits_lists = []
    #feature_lists = []
    label_lists = []
    for i, data_dict in tqdm(enumerate(data_loader), total=len(data_loader)):
        # get data
        data, label, mask, landmark = \
        data_dict['image'], data_dict['label'], data_dict['mask'], data_dict['landmark']
        label = torch.where(data_dict['label'] != 0, 1, 0)
        # move data to GPU
        data_dict['image'], data_dict['label'] = data.to(device), label.to(device)
        if mask is not None:
            data_dict['mask'] = mask.to(device)
        if landmark is not None:
            data_dict['landmark'] = landmark.to(device)

        # model forward without considering gradient computation
        predictions = inference(model, data_dict)
        label_lists += list(data_dict['label'].cpu().detach().numpy())

        # if type(model).__name__ == "UCFDetector":
        #     prediction_lists += predictions['prob']
        # else:
        #     
        prediction_lists += list(predictions['prob'].cpu().detach().numpy())
        logits_lists += list(predictions['cls'].cpu().detach().numpy())
        #feature_lists += list(predictions['feat'].cpu().detach().numpy())
    
    return np.array(prediction_lists), np.array(label_lists), np.array(logits_lists)#np.array(feature_lists)

In [6]:
def test_epoch(model, test_data_loaders, device):

    # set model to eval mode
    model.eval()

    # define test recorder
    metrics_all_datasets = {}
    preds_labels_paths = {}


    # testing for all test data
    keys = test_data_loaders.keys()
    for key in keys:
        data_dict = test_data_loaders[key].dataset.data_dict
        # compute loss for each dataset
        predictions_nps, label_nps, logits_nps = test_one_dataset(model, test_data_loaders[key], device)

        return (predictions_nps, label_nps, data_dict['image'], logits_nps)
        
        # compute metric for each dataset
        metric_one_dataset, preds_labels_paths_one_dataset = get_test_metrics(y_pred=predictions_nps, y_true=label_nps,
                                              img_names=data_dict['image'])
        metrics_all_datasets[key] = metric_one_dataset
        preds_labels_paths[key] = preds_labels_paths_one_dataset
        
        # info for each dataset
        tqdm.write(f"dataset: {key}")
        for k, v in metric_one_dataset.items():

            if k == "pred" or k == "label" or k == "paths":
                continue

            tqdm.write(f"{k}: {v}")

    #return metrics_all_datasets, preds_labels_paths

In [7]:
def init(config_path, weights_path, test_dataset, config_test):
        config = load_config(config_path)
        test_config = load_config(config_test)

        config.update(test_config)
        config["test_dataset"] = test_dataset
        config["weights_path"] = weights_path

        if config['cudnn']:
                cudnn.benchmark = True

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        init_seed(config)


        test_data_loaders = prepare_testing_data(config)

        model_class = DETECTOR[config['model_name']]
        model = model_class(config).to(device)

        ckpt = torch.load(config["weights_path"], map_location=device)

        if 'state_dict' in ckpt:
                ckpt = ckpt['state_dict']

        new_weights = {}

        for key, value in ckpt.items():
                new_key = key.replace('module.', '')
                if 'base_model.' in new_key:
                        new_key = new_key.replace('base_model.', 'backbone.')
                if 'classifier.' in new_key:
                        new_key = new_key.replace('classifier.', 'head.')
                if 'HRNet_layer.' in new_key:
                        new_key = new_key.replace('HRNet_layer.', 'backbone.')
                new_weights[new_key] = value

        if type(model).__name__ == "EffortDetector":
                model.load_state_dict(new_weights, strict=False)
        else:
                model.load_state_dict(new_weights, strict=True)
        print('===> Load checkpoint done!')

        return test_data_loaders, model, device, config


In [8]:
def get_prediction_df(predictions_nps, labels_nps, images):

    videos_real = set([p.split('/')[9] for p in images if p.split('/')[7] == 'real'])
    videos_fake = set([p.split('/')[9] for p in images if p.split('/')[7] == 'fake'])

    all_videos = videos_fake.union(videos_real)

    assert(len(videos_real) == 42)
    assert(len(videos_fake) == 42)

    df_by_frame = pd.DataFrame(images, columns=["video_path"])
    df_by_frame["video_name"] = [p.split('/')[9] for p in images]
    df_by_frame["pred"] = predictions_nps
    df_by_frame["label"] = labels_nps

    df_video_preds = pd.DataFrame(all_videos, columns=["video_name"])
    df_video_preds["label"] = df_video_preds.apply(lambda row: 1 if row["video_name"] in videos_fake else 0, axis=1)


    for video_name in all_videos:
        video_frames = df_by_frame[df_by_frame["video_name"] == video_name]

        #preds_sum = video_frames["preds"].sum()

        preds_clipped = np.clip(video_frames["pred"], 1e-7, 1 - 1e-7)
        preds_logit = logit(preds_clipped)
        avg_logit = np.mean(preds_logit)
        prediction_class_log = 0 if expit(avg_logit) < 0.5 else 1

        preds_mean = video_frames['pred'].mean()
        prediction_class_mean = 0 if preds_mean < 0.5 else 1

        if prediction_class_log != prediction_class_mean:
            print("mismatch", f"pred log {expit(avg_logit)}, pred normal {preds_mean}, class log {prediction_class_log}, class normal {prediction_class_mean}, vid {video_name}")

        df_video_preds.loc[df_video_preds["video_name"] == video_name, "class_log"] = prediction_class_log
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "class_mean"] = prediction_class_mean
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "raw_pred_log"] = expit(avg_logit)
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "raw_pred_mean"] = preds_mean
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "max_pred"] = video_frames['pred'].max()

    return df_video_preds

In [9]:
def get_prediction_df_video_level(predictions_nps, labels_nps, images):

    preds_by_video = {}
    all_videos = []
    label_by_video = {}

    for index, image in enumerate(images):
        video_name = image[0].split('/')[9]
        
        if video_name not in label_by_video.keys():
            label_by_video[video_name] = labels_nps[index]
        all_videos.append(video_name)

        if video_name in preds_by_video.keys():
            preds_by_video[video_name] = [*preds_by_video[video_name], *predictions_nps[index]]
        else:
            preds_by_video[video_name] = predictions_nps[index]


    all_videos = set(all_videos)

    assert(len(all_videos) == 84)

    df_video_preds = pd.DataFrame(all_videos, columns=["video_name"])

    df_video_preds["label"] = df_video_preds.iloc[:, 0].map(label_by_video)


    for video_name in all_videos:

        preds_clipped = np.clip(preds_by_video[video_name], 1e-7, 1 - 1e-7)
        preds_logit = logit(preds_clipped)
        avg_logit = np.mean(preds_logit)
        prediction_class_log = 0 if expit(avg_logit) < 0.5 else 1

        df_preds = pd.DataFrame(preds_by_video[video_name])
        
        preds_mean = df_preds.mean()

        prediction_class_mean = 0 if preds_mean.squeeze() < 0.5 else 1

        if prediction_class_log != prediction_class_mean:
            print("mismatch", f"pred log {expit(avg_logit)}, pred normal {preds_mean}, class log {prediction_class_log}, class normal {prediction_class_mean}, vid {video_name}")

        df_video_preds.loc[df_video_preds["video_name"] == video_name, "class_log"] = prediction_class_log
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "class_mean"] = prediction_class_mean
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "raw_pred_log"] = expit(avg_logit)
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "raw_pred_mean"] = preds_mean
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "max_pred"] = max(preds_by_video[video_name])


    return df_video_preds

In [10]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.float32):
            return obj.item()
        if isinstance(obj, np.int64):
            return obj.item()
        return super().default(obj)

In [11]:
def save_display_metrics(predictions, labels, logits, images, model_name, test_dataset, weights_path, elapsed_time, config, exps_dir):
    assert(len(predictions) == len(labels))
    assert(len(images) == len(labels))

    train_dataset = str(Path(weights_path).parent)
    print(f"Results for {model_name} ({train_dataset}) on {test_dataset} ({elapsed_time}s)")

    vid_level_models = ["i3d"]

    #Frame-level AUC

    if model_name in vid_level_models:
        clip_labels = []
        clip_preds = []

        vid_preds_mean = []
        vids_mean = {}
        for index, array in enumerate(predictions):
            frame_path = Path(images[index][0])
            video_path = str(frame_path.parent)
            for i in range(0, len(array)):
                clip_labels.append(labels[index])
            for pred in array:
                clip_preds.append(pred)

            vid_mean = np.mean(array)
            vid_preds_mean.append(vid_mean)

            vids_mean[video_path] = {
                "pred": vid_mean,
                "logit": logits[index],
                "label": labels[index]
            }

        fpr, tpr, thresholds = roc_curve(clip_labels, clip_preds)
        auc_score = roc_auc_score(clip_labels, clip_preds)

        precisions, recalls, thresholds_pr = precision_recall_curve(clip_labels, clip_preds)
        auc_pr = average_precision_score(clip_labels, clip_preds)
        print(f"(clip-level) AUC ROC: {auc_score}, AUC PR: {auc_pr}")

        #Video-level AUC
        fpr_video, tpr_video, thresholds_video = roc_curve(labels, vid_preds_mean)
        auc_score_video = roc_auc_score(labels, vid_preds_mean)
        precisions_video, recalls_video, thresholds_pr_video = precision_recall_curve(labels, vid_preds_mean)
        auc_pr_video = average_precision_score(labels, vid_preds_mean)
        print(f"(video-level) AUC ROC: {auc_score_video}, AUC PR: {auc_pr_video}")


    else:
        fpr, tpr, thresholds = roc_curve(labels, predictions)
        auc_score = roc_auc_score(labels, predictions)

        precisions, recalls, thresholds_pr = precision_recall_curve(labels, predictions)
        auc_pr = average_precision_score(labels, predictions)

        print(f"(frame-level) AUC ROC: {auc_score}, AUC PR: {auc_pr}")

        #Video-level AUC

        vids = {}

        for i, frame_str in enumerate(images):
            frame_path = Path(frame_str)
            video_path = str(frame_path.parent)
            
            if video_path in vids.keys():
                vids[video_path]["preds"].append(predictions[i])
                vids[video_path]["logits"].append(logits[i])

            else:
                vids[video_path] = {
                    "preds": [predictions[i]],
                    "logits": [logits[i]],
                    "label": labels[i]
                }

        vids_mean = {}

        labels_videos_mean = []
        preds_videos_mean = []

        for video, preds_label in vids.items():
            video_mean = np.mean(preds_label["preds"])
            left_logits = [e[0] for e in preds_label["logits"]]
            right_logits = [e[1] for e in preds_label["logits"]]
            left_mean = np.mean(left_logits)
            right_mean = np.mean(right_logits)

            vids_mean[video] = {
            "pred": video_mean,
            "logit": [left_mean, right_mean],
            "label": preds_label["label"]
                }
            
            labels_videos_mean.append(preds_label["label"])
            preds_videos_mean.append(video_mean)

        
        fpr_vid, tpr_vid, thresholds_vid = roc_curve(labels_videos_mean, preds_videos_mean)
        auc_score_video = roc_auc_score(labels_videos_mean, preds_videos_mean)

        precisions_vid, recalls_vid, thresholds_pr_vid = precision_recall_curve(labels_videos_mean, preds_videos_mean)
        auc_pr_video = average_precision_score(labels_videos_mean, preds_videos_mean)

        print(f"(video-level) AUC ROC: {auc_score_video}, AUC PR: {auc_pr_video}")

    # Saving

    metrics = {
        "model_name": model_name,
        "test_dataset": test_dataset,
        "train_dataset": train_dataset,
        "frames_per_video_test": config["frame_num"]["test"],
        "clip_size": config["clip_size"] if model_name in vid_level_models else None,
        "elapsed_time": elapsed_time,
        "frame_auc": auc_score,
        "video_auc": auc_score_video,
        "frame_auc_pr": auc_pr,
        "video_auc_pr": auc_pr_video,
    }


    results = {
        "labels": labels,
        "preds": predictions,
        "logits": logits,
        "paths": images,
    }

    this_exp_path = f"{os.path.dirname(exps_dir)}/{config["test_dataset"][0]}/{model_name}/{train_dataset}"

    os.makedirs(this_exp_path, exist_ok=True)
    os.makedirs(f"{this_exp_path}/preds/video-level", exist_ok=True)

    filename = f"{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=+1))).strftime("%Y-%m-%dT%H-%M-%SA")}.json"

    with open(f"{this_exp_path}/{filename}", "x") as file:
        json.dump(metrics, file, cls=NumpyEncoder, indent=4)

    with open(f"{this_exp_path}/preds/{filename}", "x") as file:
        json.dump(results, file, cls=NumpyEncoder, indent=4)

    video_level_results = {
        "model_name": model_name,
        "test_dataset": test_dataset,
        "train_dataset": train_dataset,
        "vids_results": vids_mean
    }

    with open(f"{this_exp_path}/preds/video-level/{filename}", "x") as file:
        json.dump(video_level_results, file, cls=NumpyEncoder, indent=4)


# Entry Point

In [13]:
exps = pd.read_csv("exps.csv")

CONFIG_TEST = "config/test_config.yaml"
CONFIG_DETECTORS = "config/detector/"
WEIGHTS_BASE = "weights/"
EXPS_DIR = "exps/"

model_name = None
config_detector_path = None
weights_path = None
test_dataset = None

predictions_nps = None
label_nps = None
images = None
elapsed_time = None
config = None

for _, row in exps.iterrows():
    model_name = row['model_name']
    config_detector_path = row['config']
    weights_path = row ['weights']
    test_dataset = row['test_dataset']
    if row["done"] == True:
        print(f"Already done {model_name} ({weights_path}) on {test_dataset}. Skip")
        continue
    print(f"Start {model_name} ({weights_path} on {test_dataset})")

    test_data_loaders, model, device, config = init(CONFIG_DETECTORS + config_detector_path, WEIGHTS_BASE + weights_path, [test_dataset], CONFIG_TEST)

    start = time.time()

    predictions_nps, label_nps, images, logits_nps = test_epoch(model, test_data_loaders, device)

    stop = time.time()
    elapsed_time = stop-start
    save_display_metrics(predictions_nps, label_nps, logits_nps, images, model_name, test_dataset, weights_path, elapsed_time, config, EXPS_DIR)

Already done capsule_net (train_on_ff-orig/capsule_best.pth) on FSAll_cdf. Skip
Already done clip_base (train_on_df40-all-ff/clip.pth) on FSAll_cdf. Skip
Already done clip_base (train_on_df40-fs-ff/clip.pth) on FSAll_cdf. Skip
Already done clip_base (train_on_df40-fr-ff/clip.pth) on FSAll_cdf. Skip
Already done clip_large (train_on_df40-all-ff/clip_large.pth) on FSAll_cdf. Skip
Already done clip_large (train_on_df40-fs-ff/clip_large.pth) on FSAll_cdf. Skip
Already done clip_large (train_on_df40-fr-ff/clip_large.pth) on FSAll_cdf. Skip
Already done core (train_on_ff-orig/core_best.pth) on FSAll_cdf. Skip
Already done efficientnetb4 (train_on_ff-orig/effnb4_best.pth) on FSAll_cdf. Skip
Already done effort (train_on_ff-orig/effort_ff_best.pth) on FSAll_cdf. Skip
Already done f3net (train_on_ff-orig/f3net_best.pth) on FSAll_cdf. Skip
Already done i3d (train_on_df40-all-ff/i3d.pth) on FSAll_cdf. Skip
Already done i3d (train_on_df40-fs-ff/i3d.pth) on FSAll_cdf. Skip
Already done i3d (train_o

100%|██████████| 4842/4842 [26:20<00:00,  3.06it/s]


Results for srm (train_on_df40-fr-ff) on FRAll_cdf (1581.1512339115143s)
(frame-level) AUC ROC: 0.8621714127945949, AUC PR: 0.9574991242171862
(video-level) AUC ROC: 0.8918947840296156, AUC PR: 0.9970110637775504
Start ucf (train_on_ff-orig/ucf_best.pth on FRAll_cdf)
LEN 2136 FRAll_Real
LEN 7770 FRAll_Fake
===> Load checkpoint done!


100%|██████████| 9684/9684 [19:43<00:00,  8.18it/s]


Results for ucf (train_on_ff-orig) on FRAll_cdf (1184.0032262802124s)
(frame-level) AUC ROC: 0.676301504848731, AUC PR: 0.8773070876787827
(video-level) AUC ROC: 0.7044329240958455, AUC PR: 0.9901758420765252
Start xception (train_on_ff-orig/xception_best.pth on FRAll_cdf)
LEN 2136 FRAll_Real
LEN 7770 FRAll_Fake
===> Load checkpoint done!


...

Results for xception (train_on_df40-all-ff) on ConfDF_norm_v2_th-m (1.9027092456817627s)
(frame-level) AUC ROC: 0.85451171875, AUC PR: 0.8625738049142978
(video-level) AUC ROC: 0.8700000000000001, AUC PR: 0.8860782191664544
Start xception (train_on_df40-fs-ff/xception.pth on ConfDF_norm_v2_th-m)
LEN 10 ConfDF_real
LEN 10 ConfDF_fake
===> Load checkpoint done!


100%|██████████| 20/20 [00:01<00:00, 12.67it/s]


Results for xception (train_on_df40-fs-ff) on ConfDF_norm_v2_th-m (1.8890466690063477s)
(frame-level) AUC ROC: 0.837998046875, AUC PR: 0.8457409279964899
(video-level) AUC ROC: 0.9, AUC PR: 0.9243181818181817
Start xception (train_on_df40-fr-ff/xception.pth on ConfDF_norm_v2_th-m)
LEN 10 ConfDF_real
LEN 10 ConfDF_fake
===> Load checkpoint done!


100%|██████████| 20/20 [00:01<00:00, 12.69it/s]


Results for xception (train_on_df40-fr-ff) on ConfDF_norm_v2_th-m (1.8838691711425781s)
(frame-level) AUC ROC: 0.7361523437499999, AUC PR: 0.758005387439237
(video-level) AUC ROC: 0.75, AUC PR: 0.8371112169564182
